# Experiment 3 — Pairwise Siamese + Frequency-Aware Labels + LOOCV

## Why the previous experiments underperformed

| | Exp1 | Exp2 | **Exp3** |
|---|---|---|---|
| Loss | Soft-label BCE per slice | ListNet + pairwise | **Pairwise Siamese (binary)** |
| Training data | 26 patients | 26 patients | **~2 028 pairs (78× more)** |
| Labels | Position sigma | Position sigma | **Physics: Δf from optimal** |
| Architecture | ResNet18/EfficientNet-B0 | ResNet18 | **EfficientNet-B0 Siamese** |
| Inference | Argmax of scores | Argmax of scores | **Win-count tournament** |
| Evaluation | 5-fold | 5-fold + LOOCV | **LOOCV (gold standard, n=26)** |
| Center-slice baseline | beats model Top-1 | beats model Top-1 | target to beat |

**Key insight**: asking "which of two slices has *better shimming quality*?" is a much simpler binary task than "score this one slice out of 13". With 26×C(13,2)=2028 pairs we have 78× more training signal. We also know each slice's exact frequency offset from the Excel — this becomes a physics-grounded label instead of a positional Gaussian.

**Runtime**: ~30–60 min with LOOCV on Kaggle GPU T4. Outputs → `exp3_outputs/`.

### Kaggle setup
Attach the same dataset as Exp1/2 and enable GPU T4.


## 0. Install dependencies

In [ ]:
import subprocess, sys

def _ensure(pkgs):
    import importlib.util
    missing = [p for p, imp in pkgs if importlib.util.find_spec(imp) is None]
    if missing:
        print("Installing:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

_ensure([
    ("pydicom",      "pydicom"),
    ("openpyxl",     "openpyxl"),
    ("pandas",       "pandas"),
    ("scikit-learn", "sklearn"),
    ("matplotlib",   "matplotlib"),
    ("seaborn",      "seaborn"),
    ("tqdm",         "tqdm"),
    ("pillow",       "PIL"),
    ("torch",        "torch"),
    ("torchvision",  "torchvision"),
])
print("Dependencies OK")


## 1. Config & paths

In [ ]:
from __future__ import annotations

import os, random, copy, json
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import pydicom
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

from sklearn.model_selection import LeaveOneOut, StratifiedKFold, train_test_split
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

try:
    from IPython.display import display
except ImportError:
    def display(x): print(x)

IS_KAGGLE = Path("/kaggle/input").exists()


def _find(root, patterns):
    for pat in patterns:
        hits = sorted(root.glob(pat))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"Not found under {root}: {patterns}")


def find_excel(root):
    return _find(root, ["**/1stcohortSAX*.xlsx", "**/*cohort*SAX*.xlsx", "**/*SAX*.xlsx"])


def find_dicom_root(root):
    hits = list(root.rglob("1st_cohort_SAX"))
    if hits:
        return hits[0]
    pats = [p for p in root.rglob("Patient-*") if p.is_dir()]
    if pats:
        return pats[0].parent
    raise FileNotFoundError(f"No DICOM cohort under {root}")


if IS_KAGGLE:
    INPUT_ROOT = Path("/kaggle/input")
    DATA_ROOT = None
    for c in sorted(INPUT_ROOT.iterdir()):
        try:
            find_dicom_root(c); DATA_ROOT = c; break
        except FileNotFoundError:
            pass
    DATA_ROOT = DATA_ROOT or next(INPUT_ROOT.iterdir())
    OUT_DIR = Path("/kaggle/working/exp3_outputs")
else:
    DATA_ROOT = Path(".").resolve()
    OUT_DIR = DATA_ROOT / "exp3_outputs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_PATH = find_excel(DATA_ROOT)
DICOM_ROOT = find_dicom_root(DATA_ROOT)

print(f"IS_KAGGLE  = {IS_KAGGLE}")
print(f"DATA_ROOT  = {DATA_ROOT}")
print(f"EXCEL_PATH = {EXCEL_PATH}")
print(f"DICOM_ROOT = {DICOM_ROOT}")
print(f"OUT_DIR    = {OUT_DIR}")


In [ ]:
@dataclass
class CFG:
    seed:              int   = 42
    img_size:          int   = 224
    batch_size:        int   = 32     # pairs per batch (fast on GPU)
    num_workers:       int   = 0
    epochs_freeze:     int   = 6
    epochs_unfreeze:   int   = 20
    lr_head:           float = 1e-3
    lr_backbone:       float = 5e-6   # very small: avoid destroying ImageNet features
    weight_decay:      float = 1e-4
    # Pairwise label: mix physics + positional
    use_freq_labels:   bool  = True   # physics-grounded soft labels
    soft_label_sigma:  float = 1.5    # fallback positional sigma
    # Auxiliary ordinal head (predict abs distance to best, λ * MSE)
    aux_lambda:        float = 0.2
    # TTA: average N forward passes per pair
    tta_n:             int   = 4
    early_stop_patience: int = 8
    loocv:             bool  = True   # gold standard for n=26
    n_folds:           int   = 5      # also run 5-fold for speed comparison
    device:            str   = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CFG()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(cfg.seed)
DEVICE = torch.device(cfg.device)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(json.dumps(asdict(cfg), indent=2))


## 2. Labels & slice index

In [ ]:
def load_labels(excel_path):
    raw = pd.read_excel(excel_path, header=None)
    for i, row in raw.iterrows():
        vals = [str(v).strip() if pd.notna(v) else "" for v in row]
        if "Anonymized ID" in vals:
            df = pd.read_excel(excel_path, header=i)
            break
    df = df.dropna(axis=1, how="all").dropna(subset=["Anonymized ID"]).reset_index(drop=True)
    df = df.rename(columns={
        "Anonymized ID": "patient_id",
        "Plane": "plane",
        "Max Freq (Hz)": "max_freq",
        "Freq Step (Hz)": "freq_step",
        "Min Freq (Hz)": "min_freq",
        "Manual frequency (Hz)": "manual_freq",
        "image no": "gt_image_no",
    })
    df["gt_image_no"] = df["gt_image_no"].astype(int)
    return df


def build_slice_index(dicom_root, labels):
    rows = []
    for _, lab in labels.iterrows():
        pid = lab["patient_id"]
        pdir = dicom_root / pid
        if not pdir.exists():
            matches = [d for d in dicom_root.iterdir()
                       if d.is_dir() and d.name.lower() == pid.lower()]
            if not matches:
                print("WARNING: missing", pid); continue
            pdir = matches[0]

        dcms = sorted(pdir.glob("*.dcm"))
        for f in dcms:
            name = f.name
            # parse instance number from filename: imgNNNN-...
            inst = None
            if name.lower().startswith("img"):
                digits = ""
                for ch in name[3:]:
                    if ch.isdigit(): digits += ch
                    else: break
                if digits: inst = int(digits)
            if inst is None:
                ds = pydicom.dcmread(f, stop_before_pixels=True)
                inst = int(getattr(ds, "InstanceNumber", 0))

            # Physics label: frequency at this slice index
            slice_freq = lab["min_freq"] + (inst - 1) * lab["freq_step"]
            dist_slices = abs(inst - int(lab["gt_image_no"]))
            dist_hz     = abs(slice_freq - lab["manual_freq"])
            soft_pos    = float(np.exp(-0.5 * (dist_slices / cfg.soft_label_sigma) ** 2))

            rows.append({
                "patient_id":  pid,
                "path":        str(f),
                "instance_no": inst,
                "gt_image_no": int(lab["gt_image_no"]),
                "manual_freq": float(lab["manual_freq"]),
                "slice_freq":  float(slice_freq),
                "freq_step":   float(lab["freq_step"]),
                "is_best":     int(inst == int(lab["gt_image_no"])),
                "dist_slices": dist_slices,
                "dist_hz":     float(dist_hz),
                "soft_pos":    soft_pos,
            })
    return pd.DataFrame(rows)


labels_df = load_labels(EXCEL_PATH)
index_df  = build_slice_index(DICOM_ROOT, labels_df)
print(labels_df[["patient_id", "gt_image_no", "manual_freq", "freq_step"]].to_string(index=False))
print(f"\nPatients={len(labels_df)}  slices={len(index_df)}  "
      f"best_ok={index_df.groupby('patient_id')['is_best'].sum().eq(1).all()}")


## 3. Frequency-aware pair generation

For each patient we generate **all C(N,2) pairs**.

**Label for pair (A, B):**
- If `use_freq_labels=True`: label = sigmoid((|dist_B| − |dist_A|) / freq_step × 2) — this encodes *"by how many steps is A better than B in Hz"*. Pure 1/0 for the best-vs-worst, soft 0.5 near ties.
- Fallback positional: Gaussian on slice distance.

**Auxiliary target**: `dist_slices_A` — predicted by a secondary head, enforces monotonicity.


In [ ]:
from itertools import combinations

def make_pairs(index_df, patient_ids, use_freq=True):
    rows = []
    for pid in patient_ids:
        g = index_df[index_df["patient_id"] == pid].sort_values("instance_no")
        slices = list(g.itertuples(index=False))
        freq_step = slices[0].freq_step if len(slices) else 25.0

        for s_a, s_b in combinations(slices, 2):
            if use_freq:
                # soft label: P(A is better than B)
                delta_hz = s_b.dist_hz - s_a.dist_hz   # positive → A is closer to optimal
                label    = float(torch.sigmoid(torch.tensor(delta_hz / max(freq_step, 1) * 2)).item())
            else:
                label = 1.0 if s_a.dist_slices < s_b.dist_slices else (
                        0.0 if s_a.dist_slices > s_b.dist_slices else 0.5)
            # also add reversed pair for balance
            rows.append({
                "patient_id": pid,
                "path_a": s_a.path, "path_b": s_b.path,
                "inst_a": s_a.instance_no, "inst_b": s_b.instance_no,
                "gt": s_a.gt_image_no,
                "label": label,
                "dist_a": s_a.dist_slices, "dist_b": s_b.dist_slices,
            })
            rows.append({
                "patient_id": pid,
                "path_a": s_b.path, "path_b": s_a.path,
                "inst_a": s_b.instance_no, "inst_b": s_a.instance_no,
                "gt": s_a.gt_image_no,
                "label": 1.0 - label,
                "dist_a": s_b.dist_slices, "dist_b": s_a.dist_slices,
            })
    return pd.DataFrame(rows)


demo_pairs = make_pairs(index_df, labels_df["patient_id"].tolist(),
                        use_freq=cfg.use_freq_labels)
print(f"Total pairs (bidirectional): {len(demo_pairs)}")
print(f"Per patient: {len(demo_pairs) // len(labels_df):.0f} avg")
print(f"Label distribution: {demo_pairs['label'].describe().round(3)}")
del demo_pairs


## 4. Pair dataset & augmentation

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def load_dicom_image(path: str) -> np.ndarray:
    arr = pydicom.dcmread(path).pixel_array.astype(np.float32)
    p1, p99 = np.percentile(arr, (1, 99))
    return np.clip((arr - p1) / max(p99 - p1, 1e-6), 0, 1)


def make_transform(train: bool, tta_seed: int = 0):
    ops = [T.Resize((cfg.img_size, cfg.img_size))]
    if train:
        ops += [
            T.RandomAffine(degrees=8, translate=(0.05, 0.05), scale=(0.93, 1.07)),
            T.ColorJitter(brightness=0.15, contrast=0.15),
            T.RandomGrayscale(p=0.05),  # mild intensity variation
        ]
    elif tta_seed > 0:
        # TTA: random small perturbations at inference
        g = torch.Generator().manual_seed(tta_seed * 1337)
        ops += [T.RandomAffine(degrees=4, translate=(0.03, 0.03),
                               scale=(0.97, 1.03))]
    ops += [T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return T.Compose(ops)


class PairDataset(Dataset):
    def __init__(self, pairs_df: pd.DataFrame, train: bool = False, tta_seed: int = 0):
        self.pairs = pairs_df.reset_index(drop=True)
        self.tf = make_transform(train, tta_seed)
        # cache images: with 331 DICOMs and 8-bit PIL they are small
        self._cache: dict[str, np.ndarray] = {}

    def _load(self, path: str) -> np.ndarray:
        if path not in self._cache:
            self._cache[path] = load_dicom_image(path)
        return self._cache[path]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        r = self.pairs.iloc[idx]
        def to_tensor(p):
            arr = (self._load(p) * 255).astype(np.uint8)
            return self.tf(Image.fromarray(arr).convert("RGB"))
        img_a = to_tensor(r["path_a"])
        img_b = to_tensor(r["path_b"])
        return {
            "img_a":    img_a,
            "img_b":    img_b,
            "label":    torch.tensor(float(r["label"]), dtype=torch.float32),
            "dist_a":   torch.tensor(float(r["dist_a"]), dtype=torch.float32),
        }


def make_pair_loader(pairs_df, train: bool, tta_seed: int = 0):
    ds = PairDataset(pairs_df, train=train, tta_seed=tta_seed)
    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=train,
        num_workers=cfg.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )


# smoke
demo_pairs = make_pairs(index_df, labels_df["patient_id"].tolist()[:4])
demo_loader = make_pair_loader(demo_pairs, train=True)
b = next(iter(demo_loader))
print("img_a", tuple(b["img_a"].shape), "label", tuple(b["label"].shape))
del demo_pairs, demo_loader, b


## 5. Siamese EfficientNet-B0 model

In [ ]:
class SiameseScorer(nn.Module):
    """
    Shared EfficientNet-B0 backbone.
    For a pair (A, B):
      score(A > B) = MLP([f_A, f_B, f_A - f_B, f_A * f_B])
    Optional aux head: predict dist_slices from f_A (ordinal regulariser).
    """

    def __init__(self):
        super().__init__()
        try:
            weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
            backbone = models.efficientnet_b0(weights=weights)
        except Exception:
            backbone = models.efficientnet_b0(pretrained=True)
        self.feat_dim = backbone.classifier[1].in_features  # 1280
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        # Pairwise comparison head
        in_dim = self.feat_dim * 4   # [fA, fB, fA-fB, fA*fB]
        self.compare_head = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(0.35),
            nn.Linear(in_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )
        # Auxiliary ordinal head (per-slice distance regressor)
        self.aux_head = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.feat_dim, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 1),
        )

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def encode(self, imgs: torch.Tensor) -> torch.Tensor:
        return self.backbone(imgs)  # [B, feat_dim]

    def compare(self, f_a: torch.Tensor, f_b: torch.Tensor) -> torch.Tensor:
        x = torch.cat([f_a, f_b, f_a - f_b, f_a * f_b], dim=1)
        return self.compare_head(x).squeeze(1)  # [B] logit

    def forward(self, img_a: torch.Tensor, img_b: torch.Tensor):
        f_a = self.encode(img_a)
        f_b = self.encode(img_b)
        logit   = self.compare(f_a, f_b)             # P(A > B)
        aux_a   = self.aux_head(f_a).squeeze(1)      # predicted dist_slices for A
        return logit, aux_a


def build_model():
    return SiameseScorer().to(DEVICE)


# smoke
m = build_model()
with torch.no_grad():
    _dummy_a = torch.randn(4, 3, 224, 224, device=DEVICE)
    _dummy_b = torch.randn(4, 3, 224, 224, device=DEVICE)
    _logit, _aux = m(_dummy_a, _dummy_b)
print(f"logit {tuple(_logit.shape)}  aux {tuple(_aux.shape)}")
print(f"Params total:    {sum(p.numel() for p in m.parameters()):,}")
print(f"Params backbone: {sum(p.numel() for p in m.backbone.parameters()):,}")
del m, _dummy_a, _dummy_b, _logit, _aux
torch.cuda.empty_cache() if DEVICE.type == "cuda" else None


## 6. Losses & inference

In [ ]:
def pairwise_loss(logit: torch.Tensor, label: torch.Tensor,
                  aux: torch.Tensor, dist_a: torch.Tensor) -> torch.Tensor:
    # Main: soft-label binary cross-entropy on comparison
    bce = F.binary_cross_entropy_with_logits(logit, label)
    # Aux: ordinal MSE (predict distance-to-best, normalised to 0-1 range)
    aux_loss = F.mse_loss(torch.sigmoid(aux), dist_a / 12.0)
    return bce + cfg.aux_lambda * aux_loss


def run_epoch(model, loader, optimizer=None):
    model.train(optimizer is not None)
    losses = []
    for b in loader:
        img_a  = b["img_a"].to(DEVICE)
        img_b  = b["img_b"].to(DEVICE)
        label  = b["label"].to(DEVICE)
        dist_a = b["dist_a"].to(DEVICE)
        with torch.set_grad_enabled(optimizer is not None):
            logit, aux = model(img_a, img_b)
            loss = pairwise_loss(logit, label, aux, dist_a)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
        losses.append(float(loss.item()))
    return float(np.mean(losses)) if losses else 0.0


@torch.no_grad()
def infer_patient_scores(model, index_df, pid: str, tta_n: int = 1) -> pd.Series:
    """
    For each slice, compute a win-rate score against all others.
    TTA: average logits over tta_n augmentation passes.
    Returns Series: instance_no -> score
    """
    model.eval()
    g = index_df[index_df["patient_id"] == pid].sort_values("instance_no").reset_index(drop=True)
    paths = list(g["path"])
    insts = list(g["instance_no"].astype(int))
    n = len(paths)

    def load_tensor(p, tta_seed=0):
        arr = (load_dicom_image(p) * 255).astype(np.uint8)
        tf = make_transform(train=False, tta_seed=tta_seed)
        return tf(Image.fromarray(arr).convert("RGB")).unsqueeze(0).to(DEVICE)

    # Pre-encode all slices (with TTA)
    feats = []  # [n, feat_dim, tta_n]
    for i in range(n):
        fs = []
        for t in range(tta_n):
            img = load_tensor(paths[i], tta_seed=t)
            fs.append(model.encode(img))  # [1, feat_dim]
        feats.append(torch.cat(fs, 0))   # [tta_n, feat_dim]

    # Win counts: for each pair (i, j), accumulate score_i
    scores = torch.zeros(n, device=DEVICE)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            logits_ij = []
            for t in range(tta_n):
                lgt = model.compare(feats[i][[t]], feats[j][[t]])  # [1]
                logits_ij.append(lgt)
            avg_logit = torch.stack(logits_ij).mean()
            scores[i] += torch.sigmoid(avg_logit)   # fractional win probability

    scores = scores.cpu().numpy()
    return pd.Series(scores, index=insts)


@torch.no_grad()
def predict_patients(model, index_df, pids, tta_n=1):
    rows = []
    for pid in pids:
        g = index_df[index_df["patient_id"] == pid]
        gt = int(g["gt_image_no"].iloc[0])
        score_series = infer_patient_scores(model, index_df, pid, tta_n=tta_n)
        for inst, sc in score_series.items():
            rows.append({"patient_id": pid, "instance_no": int(inst),
                         "gt_image_no": gt, "score": float(sc)})
    return pd.DataFrame(rows)


def patient_ranking_metrics(pred_df):
    recs = []
    for pid, g in pred_df.groupby("patient_id"):
        g = g.sort_values("score", ascending=False)
        gt = int(g["gt_image_no"].iloc[0])
        ranked = g["instance_no"].astype(int).tolist()
        p1 = ranked[0]
        rank_gt = ranked.index(gt) + 1 if gt in ranked else len(ranked)
        recs.append({
            "patient_id": pid,
            "gt_image_no": gt,
            "pred_top1": p1,
            "pred_top3": ranked[:3],
            "top1_correct": int(p1 == gt),
            "top3_correct": int(gt in ranked[:3]),
            "mae": abs(p1 - gt),
            "rank_gt": rank_gt,
        })
    rdf = pd.DataFrame(recs)
    return {
        "top1_acc":    float(rdf["top1_correct"].mean()),
        "top3_acc":    float(rdf["top3_correct"].mean()),
        "mae":         float(rdf["mae"].mean()),
        "mean_rank_gt": float(rdf["rank_gt"].mean()),
        "n_patients":  len(rdf),
        "per_patient": rdf,
    }


## 7. Train loop (freeze → unfreeze, early-stop on MAE, val on patients not pairs)

In [ ]:
def train_fold(train_pids, val_pids, test_pids, tag: str):
    print("\n" + "=" * 66)
    print(f"  {tag}   train={len(train_pids)} val={len(val_pids)} test={len(test_pids)}")
    print("=" * 66)

    # Build pair datasets
    tr_pairs = make_pairs(index_df, train_pids, use_freq=cfg.use_freq_labels)
    va_pairs = make_pairs(index_df, val_pids,   use_freq=cfg.use_freq_labels)
    tr_loader = make_pair_loader(tr_pairs, train=True)
    va_loader = make_pair_loader(va_pairs, train=False)
    print(f"  Pairs — train:{len(tr_pairs)}  val:{len(va_pairs)}")

    model = build_model()
    model.freeze_backbone()
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.lr_head, weight_decay=cfg.weight_decay,
    )
    sched = None

    best_mae, best_state = float("inf"), None
    patience = 0
    history  = []
    global_ep = 0

    for phase, n_ep in [("freeze", cfg.epochs_freeze), ("unfreeze", cfg.epochs_unfreeze)]:
        if phase == "unfreeze":
            model.unfreeze_backbone()
            opt = torch.optim.AdamW([
                {"params": model.backbone.parameters(), "lr": cfg.lr_backbone},
                {"params": model.compare_head.parameters(), "lr": cfg.lr_head * 0.3},
                {"params": model.aux_head.parameters(),     "lr": cfg.lr_head * 0.3},
            ], weight_decay=cfg.weight_decay)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
            patience = 0   # fresh patience after unfreeze
            print(f"-- unfreeze backbone @ ep {global_ep + 1}")

        for _ in range(n_ep):
            global_ep += 1
            tr_loss = run_epoch(model, tr_loader, opt)
            va_loss = run_epoch(model, va_loader)
            if sched: sched.step()

            # Val: patient-level (only val patients, no TTA for speed)
            pred = predict_patients(model, index_df, val_pids, tta_n=1)
            m = patient_ranking_metrics(pred)
            history.append({
                "ep": global_ep, "phase": phase,
                "tr": tr_loss, "va": va_loss,
                "top1": m["top1_acc"], "top3": m["top3_acc"],
                "mae": m["mae"], "rank": m["mean_rank_gt"],
            })

            improved = m["mae"] < best_mae - 1e-6
            if improved:
                best_mae   = m["mae"]
                best_state = copy.deepcopy(model.state_dict())
                patience   = 0
            else:
                patience += 1

            if global_ep == 1 or global_ep % 4 == 0 or improved:
                print(f"Ep {global_ep:02d} [{phase:8s}] "
                      f"tr={tr_loss:.4f} va={va_loss:.4f} "
                      f"Top1={m['top1_acc']:.3f} Top3={m['top3_acc']:.3f} "
                      f"MAE={m['mae']:.2f} rank={m['mean_rank_gt']:.2f}"
                      + (" *best*" if improved else ""))

            if patience >= cfg.early_stop_patience and phase == "unfreeze":
                print(f"Early stop at ep {global_ep}")
                break
        if patience >= cfg.early_stop_patience and phase == "unfreeze":
            break

    if best_state:
        model.load_state_dict(best_state)

    ckpt = OUT_DIR / f"siamese_{tag}_best.pt"
    torch.save({"model": model.state_dict(), "cfg": asdict(cfg), "tag": tag}, ckpt)
    pd.DataFrame(history).to_csv(OUT_DIR / f"history_{tag}.csv", index=False)

    # Final evaluation on test patients with TTA
    pred_test = predict_patients(model, index_df, test_pids, tta_n=cfg.tta_n)
    m_test = patient_ranking_metrics(pred_test)
    print(f"  TEST  Top1={m_test['top1_acc']:.3f}  Top3={m_test['top3_acc']:.3f}  "
          f"MAE={m_test['mae']:.2f}  rank={m_test['mean_rank_gt']:.2f}")
    return model, pd.DataFrame(history), m_test, pred_test


## 8. 5-fold patient CV (quick overview)

In [ ]:
patient_ids = np.array(labels_df["patient_id"].tolist())
y_strat     = pd.qcut(labels_df["gt_image_no"], q=3, duplicates="drop").astype(str).values

skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
fold_rows = []
oof_5fold = []

for fold, (tr_idx, te_idx) in enumerate(skf.split(patient_ids, y_strat), 1):
    tr_p  = patient_ids[tr_idx].tolist()
    te_p  = patient_ids[te_idx].tolist()
    tr_fit, va_p = (
        train_test_split(tr_p, test_size=0.2, random_state=cfg.seed + fold)
        if len(tr_p) >= 5 else (tr_p, tr_p[:1])
    )

    _, _, m_test, pred_te = train_fold(tr_fit, va_p, te_p, tag=f"fold{fold}")
    pred_te["fold"] = fold
    oof_5fold.append(pred_te)
    fold_rows.append({
        "fold": fold,
        "top1_acc": m_test["top1_acc"], "top3_acc": m_test["top3_acc"],
        "mae": m_test["mae"], "mean_rank_gt": m_test["mean_rank_gt"],
        "n_test": m_test["n_patients"],
    })
    torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

fold_df   = pd.DataFrame(fold_rows)
oof_5_df  = pd.concat(oof_5fold, ignore_index=True)
fold_df.to_csv(OUT_DIR / "exp3_kfold_results.csv", index=False)
oof_5_df.to_csv(OUT_DIR / "exp3_kfold_oof_scores.csv", index=False)

print("\n=== 5-fold summary ===")
display(fold_df)
print(fold_df[["top1_acc", "top3_acc", "mae", "mean_rank_gt"]].agg(["mean", "std"]).round(3))
kfold_oof_m = patient_ranking_metrics(oof_5_df)
print("\nPooled OOF:", {k: round(kfold_oof_m[k], 3) for k in ["top1_acc", "top3_acc", "mae", "mean_rank_gt"]})


## 9. LOOCV — leave-one-patient-out (gold standard for n=26)

In [ ]:
loo_rows   = []
loo_preds  = []

if cfg.loocv:
    for i, pid in enumerate(patient_ids, 1):
        tr_p  = [p for p in patient_ids if p != pid]
        te_p  = [pid]
        tr_fit, va_p = train_test_split(tr_p, test_size=0.15, random_state=cfg.seed + i)

        _, _, m_te, pred_te = train_fold(tr_fit, va_p, te_p, tag=f"loo{i:02d}_{pid}")
        pred_te["loo_idx"] = i
        loo_preds.append(pred_te)
        loo_rows.append({
            "fold": i, "test_pid": pid,
            "top1_acc": m_te["top1_acc"], "top3_acc": m_te["top3_acc"],
            "mae": m_te["mae"], "rank_gt": int(m_te["per_patient"]["rank_gt"].iloc[0]),
        })
        print(f"LOO {i:02d}/{len(patient_ids)} {pid}: "
              f"top1={m_te['top1_acc']:.2f} top3={m_te['top3_acc']:.2f} mae={m_te['mae']:.2f}")
        torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

    loo_df  = pd.DataFrame(loo_rows)
    loo_oof = pd.concat(loo_preds, ignore_index=True)
    loo_df.to_csv(OUT_DIR / "exp3_loocv_results.csv", index=False)
    loo_oof.to_csv(OUT_DIR / "exp3_loocv_oof_scores.csv", index=False)

    print("\n=== LOOCV summary ===")
    display(loo_df)
    print(loo_df[["top1_acc", "top3_acc", "mae", "rank_gt"]].agg(["mean", "std"]).round(3))
    m_loo = patient_ranking_metrics(loo_oof)
    print("\nLOOCV pooled:", {k: round(m_loo[k], 3)
                              for k in ["top1_acc", "top3_acc", "mae", "mean_rank_gt"]})
    display(m_loo["per_patient"].sort_values("top1_correct", ascending=False))
else:
    print("LOOCV skipped (cfg.loocv=False).")
    loo_df = loo_oof = None


## 10. Comprehensive comparison — all experiments

In [ ]:
# All-patient baselines
ns = index_df.groupby("patient_id").size().values

def all_patient_baseline(strategy):
    rows = []
    mode_gt = int(labels_df["gt_image_no"].mode().iloc[0])
    for pid, g in index_df.groupby("patient_id"):
        inst = sorted(g["instance_no"].tolist())
        gt = int(g["gt_image_no"].iloc[0])
        pred = inst[len(inst) // 2] if strategy == "center" else (
               mode_gt if mode_gt in inst else inst[len(inst) // 2])
        neigh = sorted(inst, key=lambda i: (abs(i - pred), i))[:3]
        rows.append({"top1": int(pred == gt), "top3": int(gt in neigh), "mae": abs(pred - gt)})
    r = pd.DataFrame(rows)
    return {"top1_acc": float(r.top1.mean()), "top3_acc": float(r.top3.mean()), "mae": float(r.mae.mean())}

rows = [
    # Exp3 LOOCV
    *([{
        "exp": "Exp3 Siamese EfficientNet-B0 (LOOCV)",
        "top1_acc": patient_ranking_metrics(loo_oof)["top1_acc"],
        "top3_acc": patient_ranking_metrics(loo_oof)["top3_acc"],
        "mae":      patient_ranking_metrics(loo_oof)["mae"],
        "eval":     "LOOCV n=26",
    }] if loo_oof is not None else []),
    # Exp3 5-fold OOF
    {
        "exp": "Exp3 Siamese EfficientNet-B0 (5-fold OOF)",
        "top1_acc": kfold_oof_m["top1_acc"],
        "top3_acc": kfold_oof_m["top3_acc"],
        "mae":      kfold_oof_m["mae"],
        "eval":     "5-fold OOF",
    },
    # Exp2 LOOCV results (from run2 paper)
    {"exp": "Exp2 ResNet18 ListNet (LOOCV)",
     "top1_acc": 0.038, "top3_acc": 0.462, "mae": 2.19, "eval": "LOOCV n=26"},
    {"exp": "Exp1 ResNet18 BCE (5-fold)",
     "top1_acc": 0.227, "top3_acc": 0.613, "mae": 1.35, "eval": "5-fold mean"},
    # Baselines
    {"exp": "Center slice (naive)",  "eval": "all patients", **{k: v for k,v in all_patient_baseline("center").items()}},
    {"exp": "Global-mode GT (naive)","eval": "all patients", **{k: v for k,v in all_patient_baseline("mode").items()}},
    {"exp": "Random (expected)",
     "top1_acc": float(np.mean(1/ns)), "top3_acc": float(np.mean([min(3,n)/n for n in ns])),
     "mae": float(np.nan), "eval": "analytic"},
]

compare_df = pd.DataFrame(rows)
compare_df.to_csv(OUT_DIR / "exp3_all_experiment_comparison.csv", index=False)
print("=== All-experiment comparison ===")
display(compare_df.round(3))


## 11. Plots

In [ ]:
# Bar chart: comparison across experiments
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(compare_df))
w = 0.35
ax.bar(x - w/2, compare_df["top1_acc"].fillna(0), width=w, label="Top-1")
ax.bar(x + w/2, compare_df["top3_acc"].fillna(0), width=w, label="Top-3")
ax.set_xticks(x)
ax.set_xticklabels(compare_df["exp"], rotation=30, ha="right", fontsize=8)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_title("Best-slice selection — all experiments vs baselines")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "exp3_all_experiment_bars.png", dpi=140, bbox_inches="tight")
plt.show()

# LOOCV per-patient error heatmap
if loo_oof is not None:
    m_pp = patient_ranking_metrics(loo_oof)["per_patient"].set_index("patient_id")
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    mae_s = m_pp["mae"].sort_values()
    axes[0].barh(mae_s.index, mae_s.values, color=["green" if v <= 1 else ("orange" if v <= 2 else "red") for v in mae_s.values])
    axes[0].axvline(mae_s.mean(), color="k", ls="--", lw=1)
    axes[0].set_title("LOOCV MAE per patient")
    axes[0].set_xlabel("MAE (slice units)")
    rk_s = m_pp["rank_gt"].sort_values()
    axes[1].barh(rk_s.index, rk_s.values)
    axes[1].axvline(rk_s.mean(), color="k", ls="--", lw=1)
    axes[1].set_title("GT rank per patient (1 = correct)")
    axes[1].set_xlabel("rank")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "exp3_loocv_per_patient.png", dpi=140, bbox_inches="tight")
    plt.show()

# 5-fold bars
fig, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(len(fold_df))
ax.bar(x - w/2, fold_df["top1_acc"], width=w, label="Top-1")
ax.bar(x + w/2, fold_df["top3_acc"], width=w, label="Top-3")
ax.axhline(fold_df["top1_acc"].mean(), color="C0", ls="--", lw=1)
ax.axhline(fold_df["top3_acc"].mean(), color="C1", ls="--", lw=1)
ax.set_xticks(x); ax.set_xticklabels([f"fold{i}" for i in fold_df["fold"]])
ax.set_ylim(0, 1.05); ax.set_ylabel("Accuracy")
ax.set_title("Exp3 Siamese — 5-fold CV"); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "exp3_kfold_bars.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
# Visualize best LOOCV patient predictions
if loo_oof is not None:
    loo_pp = patient_ranking_metrics(loo_oof)["per_patient"]
    show  = loo_pp[loo_pp["top1_correct"] == 1]["patient_id"].head(2).tolist()
    show += loo_pp.sort_values("mae", ascending=False).head(2)["patient_id"].tolist()
    show  = list(dict.fromkeys(show))[:4]

    for pid in show:
        sub = loo_oof[loo_oof["patient_id"] == pid].sort_values("instance_no")
        gt  = int(sub["gt_image_no"].iloc[0])
        ranked = sub.sort_values("score", ascending=False)["instance_no"].astype(int).tolist()
        paths  = index_df[index_df["patient_id"] == pid].set_index("instance_no")["path"]
        n = len(sub); cols = min(7, n); rows = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(2.1 * cols, 2.4 * rows))
        axes = np.array(axes).reshape(-1)
        for ax, (_, r) in zip(axes, sub.iterrows()):
            ax.imshow(load_dicom_image(paths.loc[int(r["instance_no"])]), cmap="gray")
            inst  = int(r["instance_no"])
            flags = (["GT"] if inst == gt else []) + (["P1"] if inst == ranked[0] else (["P3"] if inst in ranked[:3] else []))
            ax.set_title(f"#{inst} {r['score']:.1f}
{'/'.join(flags)}", fontsize=8)
            col = "lime" if inst == gt else ("dodgerblue" if inst == ranked[0] else ("orange" if inst in ranked[:3] else "0.3"))
            for sp in ax.spines.values(): sp.set_color(col); sp.set_linewidth(2)
            ax.axis("off")
        for ax in axes[n:]: ax.axis("off")
        fig.suptitle(f"LOOCV | {pid}  GT={gt}  pred={ranked[0]}  top3={ranked[:3]}", y=1.02)
        plt.tight_layout(); plt.show()


---
## Outputs in `exp3_outputs/`
| File | Content |
|------|---------|
| `siamese_fold*_best.pt` | 5-fold model checkpoints |
| `siamese_loo*_best.pt`  | LOOCV model checkpoints |
| `exp3_kfold_results.csv` | per-fold metrics |
| `exp3_loocv_results.csv` | per-patient LOOCV metrics |
| `exp3_loocv_oof_scores.csv` | raw pairwise win-scores per patient |
| `exp3_all_experiment_comparison.csv` | Exp1/2/3 + baselines table |
| `*.png` | bar charts, per-patient MAE, slice visualisations |

## Architecture recap

```
DICOM slice A  ──┐
                 ├─► EfficientNet-B0 (shared) ─► f_A
DICOM slice B  ──┘                              f_B
                    Compare head([f_A, f_B, f_A-f_B, f_A*f_B]) ─► P(A > B)
                    Aux head(f_A)  ─► predicted dist_slices (ordinal regulariser)
```

**Training**: frequency-aware soft labels (Δf in Hz → soft 0/1),  
freeze 6ep → unfreeze with backbone lr 5e-6, early-stop on val patient MAE.  
**Inference**: for each slice, sum win-probabilities against all others → win-count score → argmax = top-1.  
TTA: repeat 4 passes with random affine jitter, average logits.
